#### 1. Setup

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append("../src")
from column_aliases import COLUMN_ALIASES

INPUT_FILE  = Path("../data/raw/mental-health-in-tech-2016_20161114.csv")
OUTPUT_FILE = Path("../data/processed/preprocessed.csv")

print("Setup abgeschlossen.")


Setup abgeschlossen.


#### 2. Column Aliases importieren

In [2]:
df = pd.read_csv(INPUT_FILE)
print(f"Geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")

rename_map = {k: v for k, v in COLUMN_ALIASES.items() if k in df.columns}
df.rename(columns=rename_map, inplace=True)
if "Why or why not?.1" in df.columns:
    df.rename(columns={"Why or why not?.1": "interview_mental_why"}, inplace=True)

print(f"Nach Umbenennung: {df.shape[1]} Spalten")
df.head(2)


Geladen: 1433 Zeilen, 63 Spalten
Nach Umbenennung: 63 Spalten


,self_employed,company_size,tech_company,tech_role,mh_benefits,know_options,employer_discussed_mh,employer_resources,anonymity_protected,leave_difficulty,...,interferes_treated,interferes_untreated,age,gender,country_live,us_state_live,country_work,us_state_work,work_position,remote_work
0,0,26-100,1.0,NaN,Not eligible for coverage / N/A,NaN,No,No,I don't know,Very easy,...,Not applicable to me,Not applicable to me,39,Male,United Kingdom,NaN,United Kingdom,NaN,Back-end Developer,Sometimes
1,0,6-25,1.0,NaN,No,Yes,Yes,Yes,Yes,Somewhat easy,...,Rarely,Sometimes,29,male,United States of America,Illinois,United States of America,Illinois,Back-end Developer|Front-end Developer,Never


#### 3. Alter bereinigen

In [3]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")

invalid = df[(df["age"] < 18) | (df["age"] > 80)]
print(f"Entfernte Ausreißer ({len(invalid)} Zeilen): {sorted(invalid['age'].dropna().unique().tolist())}")

df = df[(df["age"] >= 18) & (df["age"] <= 80)].copy()
print(f"Shape nach Altersbereinigung: {df.shape}")


Entfernte Ausreißer (5 Zeilen): [3, 15, 17, 99, 323]
Shape nach Altersbereinigung: (1428, 63)


#### 4. Gender bereinigen

In [4]:
GENDER_MAP = {
    # Male
    "male": "Male", "m": "Male", "male ": "Male", "male.": "Male",
    "cis male": "Male", "cis man": "Male", "cisdude": "Male",
    "male (cis)": "Male", "man": "Male", "dude": "Male",
    "sex is male": "Male", "malr": "Male", "mail": "Male",
    "m|": "Male", "male (trans, ftm)": "Male",
    "i'm a man why didn't you make this a drop down question. you should of asked sex? and i would of answered yes please. seriously how much text can this take?": "Male",
    "male 9:1 female, roughly": "Male",
    # Female
    "female": "Female", "f": "Female", "female ": "Female",
    "cis female": "Female", "cis-woman": "Female", "woman": "Female",
    "cisgender female": "Female", "fem": "Female", "fm": "Female",
    "female/woman": "Female", "female assigned at birth": "Female",
    "i identify as female.": "Female",
    "female (props for making this a freeform field, though)": "Female",
    "female-bodied; no feelings about gender": "Female",
    "transitioned, m2f": "Female", "mtf": "Female",
    "transgender woman": "Female",
    "female or multi-gender femme": "Female",
    # Non-binary
    "non-binary": "Non-binary", "nonbinary": "Non-binary",
    "genderqueer": "Non-binary", "genderqueer woman": "Non-binary",
    "genderfluid": "Non-binary", "genderfluid (born female)": "Non-binary",
    "genderflux demi-girl": "Non-binary", "agender": "Non-binary",
    "androgynous": "Non-binary", "bigender": "Non-binary",
    "enby": "Non-binary", "fluid": "Non-binary", "queer": "Non-binary",
    "nb masculine": "Non-binary", "other": "Non-binary",
    "other/transfeminine": "Non-binary", "male/genderqueer": "Non-binary",
    "afab": "Non-binary", "human": "Non-binary",
    "none of your business": "Non-binary", "unicorn": "Non-binary",
}

before_unique = df["gender"].nunique()
df["gender"] = df["gender"].apply(
    lambda val: GENDER_MAP.get(str(val).strip().lower(), "Non-binary") if pd.notna(val) else np.nan
)
print(f"Gender: {before_unique} Rohwerte -> {df['gender'].nunique()} Kategorien")
print(df["gender"].value_counts())


Gender: 69 Rohwerte -> 3 Kategorien
gender
Male          1055
Female         343
Non-binary      27
Name: count, dtype: int64


#### 5. Country verdichten → Region

In [5]:
REGION_MAP = {
    "United States of America": "North America", "Canada": "North America",
    "Mexico": "Latin America",
    "United Kingdom": "Europe", "Germany": "Europe", "Netherlands": "Europe",
    "France": "Europe", "Sweden": "Europe", "Ireland": "Europe",
    "Switzerland": "Europe", "Denmark": "Europe", "Finland": "Europe",
    "Belgium": "Europe", "Italy": "Europe", "Poland": "Europe",
    "Spain": "Europe", "Austria": "Europe", "Romania": "Europe",
    "Czech Republic": "Europe", "Norway": "Europe", "Bulgaria": "Europe",
    "Lithuania": "Europe", "Estonia": "Europe", "Bosnia and Herzegovina": "Europe",
    "Slovakia": "Europe", "Greece": "Europe", "Hungary": "Europe", "Serbia": "Europe",
    "Australia": "Oceania", "New Zealand": "Oceania",
}

df["region"] = df["country_live"].apply(lambda c: REGION_MAP.get(c, "Other") if pd.notna(c) else "Other")
print(f"Länder: {df['country_live'].nunique()} -> Regionen: {df['region'].nunique()}")
print(df["region"].value_counts())
df.drop(columns=["country_live"], inplace=True)


Länder: 53 -> Regionen: 5
region
North America    914
Europe           409
Other             59
Oceania           44
Latin America      2
Name: count, dtype: int64


#### 6. Fehlende Werte in Arbeitgeberfragen

In [6]:
EMPLOYER_COLUMNS = [
    "company_size", "tech_company", "mh_benefits", "know_options",
    "employer_discussed_mh", "employer_resources", "anonymity_protected",
    "leave_difficulty", "neg_consequences_employer", "neg_consequences_physical",
    "comfortable_coworkers", "comfortable_supervisor",
    "employer_takes_mh_serious", "observed_neg_consequences",
]
PREV_EMPLOYER_COLUMNS = [
    "prev_mh_benefits", "prev_aware_options", "prev_discussed_mh",
    "prev_resources", "prev_anonymity", "prev_neg_consequences",
    "prev_neg_physical", "prev_comfortable_coworkers",
    "prev_comfortable_supervisor", "prev_mh_serious", "prev_observed_neg",
]

self_emp_mask = df["self_employed"] == 1
for col in EMPLOYER_COLUMNS:
    if col in df.columns:
        df[col] = df[col].astype(object)
        df.loc[self_emp_mask, col] = np.nan

no_prev_mask = df["has_previous_employers"] == 0
for col in PREV_EMPLOYER_COLUMNS:
    if col in df.columns:
        df.loc[no_prev_mask, col] = np.nan

print(f"Selbstständige: {self_emp_mask.sum()} Zeilen mit NaN in Arbeitgeber-Spalten gesetzt")
print(f"Ohne frühere AG: {no_prev_mask.sum()} Zeilen mit NaN in Prev-Spalten gesetzt")


Selbstständige: 286 Zeilen mit NaN in Arbeitgeber-Spalten gesetzt
Ohne frühere AG: 168 Zeilen mit NaN in Prev-Spalten gesetzt


#### 7. Spalten entfernen: Freitext und fehlende Werte

In [7]:
DROP_COLUMNS = [
    "interview_physical_why", "interview_mental_why",
    "diagnosed_conditions", "believed_conditions",
    "professional_diagnosis_detail",
    "us_state_live", "us_state_work",
    "country_work",
    "tech_role",
    "medical_coverage",
    "know_resources",
    "reveal_clients", "reveal_clients_impact",
    "reveal_coworkers", "reveal_coworkers_impact",
    "productivity_affected", "pct_work_affected",
    "observations_less_likely",
    "Do you have medical coverage (private insurance or state-provided) which includes treatment of \xa0mental health issues?",
]

cols_to_drop = [c for c in DROP_COLUMNS if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)
print(f"{len(cols_to_drop)} Spalten entfernt. Shape: {df.shape}")


18 Spalten entfernt. Shape: (1428, 45)


#### 8. Freitext work_position in binäre Spalten umwandeln

In [8]:
positions = set()
for val in df["work_position"].dropna():
    for pos in str(val).split("|"):
        positions.add(pos.strip())

pos_counts = {
    pos: df["work_position"].dropna().apply(lambda x: pos in str(x)).sum()
    for pos in positions
}
kept = {p: c for p, c in pos_counts.items() if c >= 30}

for pos in sorted(kept.keys()):
    col_name = f"pos_{pos.lower().replace(' ', '_').replace('/', '_').replace('-', '_')}"
    df[col_name] = df["work_position"].apply(
        lambda x: 1 if pd.notna(x) and pos in str(x) else 0
    )

df.drop(columns=["work_position"], inplace=True)
print(f"{len(positions)} Positionen gefunden, {len(kept)} als Features behalten (>= 30 Nennungen):")
for pos, count in sorted(kept.items(), key=lambda x: x[1], reverse=True):
    print(f"  {pos}: {count}")


12 Positionen gefunden, 11 als Features behalten (>= 30 Nennungen):
  Back-end Developer: 735
  Front-end Developer: 501
  DevOps/SysAdmin: 282
  Supervisor/Team Lead: 276
  Other: 186
  Support: 168
  One-person shop: 161
  Designer: 135
  Executive Leadership: 101
  Dev Evangelist/Advocate: 99
  Sales: 31


#### 9. Spalten in numerische Werte umwandeln
- Yes/No/Maybe -> 2/1/0
- observed_bad_response -> 0–3

In [9]:
# observed_bad_response
obs_map = {"No": 0, "Maybe/Not sure": 1, "Yes, I observed": 2, "Yes, I experienced": 3}
if "observed_bad_response" in df.columns:
    df["observed_bad_response"] = df["observed_bad_response"].map(obs_map)

# Ordinal
ORDINAL_MAPS = {
    "leave_difficulty": {"Very difficult": 1, "Somewhat difficult": 2,
        "Neither easy nor difficult": 3, "I don't know": 3, "Somewhat easy": 4, "Very easy": 5},
    "share_friends_family": {"Not open at all": 1, "Somewhat not open": 2,
        "Neutral": 3, "Not applicable to me (I do not have a mental illness)": 3,
        "Somewhat open": 4, "Very open": 5},
    "interferes_treated": {"Often": 1, "Sometimes": 2, "Rarely": 3, "Never": 4,
        "Not applicable to me": np.nan},
    "interferes_untreated": {"Often": 1, "Sometimes": 2, "Rarely": 3, "Never": 4,
        "Not applicable to me": np.nan},
    "company_size": {"1-5": 1, "6-25": 2, "26-100": 3, "100-500": 4,
        "500-1000": 5, "More than 1000": 6},
}
for col, mapping in ORDINAL_MAPS.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)

# Yes/No/Maybe
YES_NO_MAYBE_MAP = {"Yes": 2, "Maybe": 1, "No": 0}
YES_NO_MAYBE_COLUMNS = [
    "neg_consequences_employer", "neg_consequences_physical",
    "comfortable_coworkers", "comfortable_supervisor",
    "interview_physical", "interview_mental",
    "current_disorder", "past_disorder",
]
for col in YES_NO_MAYBE_COLUMNS:
    if col in df.columns:
        df[col] = df[col].map(YES_NO_MAYBE_MAP)

# Yes/No/I don't know
YES_NO_DONTKNOW_MAP = {"Yes": 2, "I don't know": 1, "No": 0}
YES_NO_DONTKNOW_COLUMNS = [
    "employer_discussed_mh", "employer_resources", "anonymity_protected",
    "employer_takes_mh_serious", "family_history",
]
for col in YES_NO_DONTKNOW_COLUMNS:
    if col in df.columns:
        df[col] = df[col].map(YES_NO_DONTKNOW_MAP)

# Einzelne Spalten
df["mh_benefits"] = df["mh_benefits"].map({
    "No": 0, "Not eligible for coverage / N/A": 0,
    "I don't know": 1, "Yes": 2, "N/A": np.nan})
df["know_options"] = df["know_options"].map({"No": 0, "I am not sure": 1, "Yes": 2})
df["mh_hurts_career"] = df["mh_hurts_career"].map({
    "No, it has not": 0, "No, I don't think it would": 1,
    "Maybe": 2, "Yes, I think it would": 3, "Yes, it has": 4})
df["coworkers_view_neg"] = df["coworkers_view_neg"].map({
    "No, they do not": 0, "No, I don't think they would": 1,
    "Maybe": 2, "Yes, I think they would": 3, "Yes, they do": 4})
df["observed_neg_consequences"] = df["observed_neg_consequences"].map({"Yes": 1, "No": 0})
df["professionally_diagnosed"] = df["professionally_diagnosed"].map({"Yes": 1, "No": 0})

print("Kodierung abgeschlossen.")
print(df.dtypes.value_counts())


Kodierung abgeschlossen.
int64      24
float64    16
str        14
object      1
Name: count, dtype: int64


#### 10. Eigenschaften von früheren Arbeitgebern

In [10]:
prev_mappings = {
    "prev_mh_benefits": {"No, none did": 0, "I don't know": 1, "Some did": 2, "Yes, they all did": 3, "N/A": np.nan},
    "prev_aware_options": {"N/A (not currently aware)": 0, "No, I only became aware later": 1,
        "I was aware of some": 2, "Yes, I was aware of all of them": 3, "N/A": np.nan},
    "prev_discussed_mh": {"None did": 0, "I don't know": 1, "Some did": 2, "Yes, they all did": 3, "N/A": np.nan},
    "prev_resources": {"None did": 0, "I don't know": 1, "Some did": 2, "Yes, they all did": 3, "N/A": np.nan},
    "prev_anonymity": {"No": 0, "I don't know": 1, "Sometimes": 2, "Yes, always": 3, "N/A": np.nan},
    "prev_neg_consequences": {"None of them": 0, "I don't know": 1, "Some of them": 2, "Yes, all of them": 3, "N/A": np.nan},
    "prev_neg_physical": {"None of them": 0, "I don't know": 1, "Some of them": 2, "Yes, all of them": 3, "N/A": np.nan},
    "prev_comfortable_coworkers": {"No, at none of my previous employers": 0,
        "Some of my previous employers": 1, "Yes, at all of my previous employers": 2, "N/A": np.nan},
    "prev_comfortable_supervisor": {"No, at none of my previous employers": 0, "I don't know": 1,
        "Some of my previous employers": 2, "Yes, at all of my previous employers": 3, "N/A": np.nan},
    "prev_mh_serious": {"None did": 0, "I don't know": 1, "Some did": 2, "Yes, they all did": 3, "N/A": np.nan},
    "prev_observed_neg": {"None of them": 0, "Some of them": 1, "Yes, all of them": 2, "N/A": np.nan},
}

for col, mapping in prev_mappings.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)

print("Prev-Employer-Kodierung abgeschlossen.")


Prev-Employer-Kodierung abgeschlossen.


#### 11. Gender / Remote Work / Region

In [11]:
cat_cols = [c for c in ["gender", "remote_work", "region"] if c in df.columns]
before_cols = len(df.columns)
df = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)
print(f"One-Hot-Encoding: {len(cat_cols)} Spalten -> {len(df.columns) - before_cols + len(cat_cols)} neue Spalten")
print(f"Shape: {df.shape}")


One-Hot-Encoding: 3 Spalten -> 11 neue Spalten
Shape: (1428, 63)


#### 12. NaN -> Median, `tech_company` bleibt NaN

In [12]:
na_before = df.isna().sum().sum()
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    if df[col].isna().any() and col != "tech_company":
        median_val = df[col].median()
        n_filled = df[col].isna().sum()
        df[col] = df[col].fillna(median_val)

na_after = df.isna().sum().sum()
print(f"NaN gesamt: {na_before} -> {na_after}")
print(f"Verbleibende NaN pro Spalte (nur > 0):")
remaining = df.isna().sum()
print(remaining[remaining > 0])


NaN gesamt: 7093 -> 286
Verbleibende NaN pro Spalte (nur > 0):
tech_company    286
dtype: int64


#### 13. Datei erstellen

In [13]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)

print(f"Gespeichert: {OUTPUT_FILE}")
print(f"Shape: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
print(f"NaN gesamt: {df.isna().sum().sum()}")
print(f"Datentypen:\n{df.dtypes.value_counts()}")


Gespeichert: ..\data\processed\preprocessed.csv
Shape: 1428 Zeilen, 63 Spalten
NaN gesamt: 286
Datentypen:
int64      35
float64    27
object      1
Name: count, dtype: int64
